---

## 📄 Technical report: Classification of Archaeological Cereals Using Stable Isotopes

**Author:** David Larreina-García  
**Date:** 15/06/2026  
**Repository:** [github.com/dplauto-cpu/Carpology](https://github.com/dplauto-cpu/Carpology)

---

## 1. Introduction

This project is being carried out as part of an assessed practical component of a Machine Learning course. While the focus is not on carpological research, datasets from academic research have been used, and basic guidance has been provided by the **isoTOPIK** laboratory at the *University of Burgos* (UBU). This is a preliminary study that is expected to be revisited at some point from a purely archaeological perspective.

### 1.1 Archaeological Context
The domestication of cereals (wheat and barley) during the Neolithic revolution transformed human societies. However, identifying charred archaeological seeds to species level is seriously challenging due to degradatation and carbonisation. Stable isotope analysis (δ13C and δ15N) offers an alternative: δ13C indicates water stress, while δ15N reflects aridity or manuring intensity.

### 1.2 Research Question
Is it possible to train a ML model to distinguish barley vs wheat isotopes, and integrate the data into a coherent geographical and chronological archaeological framework?

### 1.3 Objectives
- **Primary:** Build a supervised model to classify barley vs wheat
- **Secondary:** Identify isotopic clusters using unsupervised learning
- **Tertiary:** Create interactive maps for archaeological interpretation

---

## 2. Dataset

### 2.1 Source and Composition
The dataset `Med_plants.csv` (1341 records, 20 fields) merges:
- **MAIA database** (Mediterranean Archive of Isotopic dAta)
- **IsoTOPIKLab dataset** (University of Burgos)
- 1147 samples of barley and wheat, the rest of plants unused in the ML

### 2.2 Target Variable Distribution
| Class | Count | Percentage |
|:---|:---|:---|
| Barley | 586 | 51.1% |
| Wheat | 561 | 48.9% |

### 2.3 Main Features

| Feature | Type | Description |
|:---|:---|:---|
| `IRMS_d13C_Collagen` | Continuous | Carbon isotope (‰) |
| `d15N_Collagen` | Continuous | Nitrogen isotope (‰) |
| `Latitude_N` | Continuous | Latitude (decimal degrees) |
| `Longitude_E` | Continuous | Longitude (decimal degrees) |
| `Chronological_Period_clean` | Categorical | Neolithic, Bronze Age, Iron Age |
| `Mediterranean_Basin` | Categorical | Western, Central, Eastern, Undefined |
---  
| Cultural Period              | Mean date (BP) | BCE/CE range (approx.) |
|--------------------|----------------|-------------------------|
| Neolithic          | ≥ 8000         | ≥ 6000 BCE              |
| Neolithic_Late     | 6000 – 7999    | 4000 – 5999 BCE         |
| Chalcolithic       | 5200 – 5999    | 3200 – 3999 BCE         |
| Early_Bronze       | 4200 – 5199    | 2200 – 3199 BCE         |
| Middle_Late_Bronze | 3500 – 4199    | 1500 – 2199 BCE         |
| Iron_Age_Early     | 3000 – 3499    | 1000 – 1499 BCE         |
| Iron_Age_Late      | 2400 – 2999    | 500 – 999 BCE           |
| Iron_Roman         | 1700 – 2399    | 500 BCE – 250 CE        |
---
| Region            | Longitude (decimal degrees) | Latitude (decimal degrees) |
|-------------------|-----------------------------|----------------------------|
| General limits    | -10 ≤ lon ≤ 42              | 25 ≤ lat ≤ 46              |
| Western_Med       | -10 ≤ lon ≤ 11              | 25 ≤ lat ≤ 46              |
| Central_Med       | 11 < lon ≤ 23               | 25 ≤ lat ≤ 46              |
| Eastern_Med       | 23 < lon ≤ 42               | 25 ≤ lat ≤ 46              |
---

## 3. Methodology

### 3.1 Preprocessing

# Steps applied
1. Filter: keep only 'Hordeum vulgare' and 'Triticum dicoccum'
2. Encode categorical variables (LabelEncoder)
3. Scale numerical features (StandardScaler)
4. No missing values in key features

### 3.2 Dataset Creation Process

The final dataset `Med_Plants.csv` (1341 records, 20 fields) was created through a multi-step pipeline:

#### Step 1: Data integration

- **MAIA database** (2,323 samples, 93 columns): extracted relevant columns including `Entry_ID`, `Site_Name`, `IRMS_d13C_Collagen`, `d15N_Collagen`, `Latitude_N`, `Longitude_E`, `Genus`, `Species`, `Cultural_Horizon`, `Absolute_Chronology`
- **Trigo_cebada dataset** (3,344 samples, 17 columns): extracted `site`, `latitud`, `longitud`, `cereal`, `taxon`, `Cultural period`, `d13C`, `d15N`
- **Merge strategy:** Left join on `Entry_ID` where available; for remaining samples, approximate join by coordinates (KDTree with 0.2° tolerance, ~22 km)

#### Step 2: Initial cleaning
- Removed samples without both isotopic values (δ13C and δ15N)
- Result: 1,346 samples retained (42% removed due to missing isotopes)

#### Step 3: Feature engineering
- **Chronological periods:** Extracted from `Absolute_Chronology` using regex patterns, converted to BP dates, then assigned to 8 standardised periods (Neolithic to Iron Age). Dates were calibrated using the conversion: BP = BCE + 1950.
- **Mediterranean basins:** Assigned based on longitude thresholds (Western: lon ≤ 11°, Central: 11° < lon ≤ 23°, Eastern: 23° < lon ≤ 42°)
- **Cereal classification:** Created `Cereal_Type` column distinguishing barley, wheat, broomcorn millet, oat, and non-cereals
- **Pulse classification:** Created `Pulse_Type` for legumes (lentil, faba bean, grass pea, pea, vetch) and other crops (olive, flax, pistachio)
- **Unified site names:** Combined `Site_Name` (MAIA) and `site` (trigo_cebada) into `Archaeological_Site` (98.7% coverage)

#### Step 4: Final filtering
- Removed 5 samples with undefined Mediterranean basin
- **Final dataset:** 1,341 samples × 20 columns

### 3.3 Exploratory Data Analysis (EDA)

The following analyses were conducted:

**Isotopic distributions by period:**
- Boxplots of δ13C and δ15N for each chronological period revealed a progressive trend toward more positive δ13C (increasing water stress) from Neolithic to Iron Age
- δ15N showed greater variability, with higher values in the Bronze Age suggesting possible manuring intensification

**Isotopic scatter by period:**
- Overlapping isotopic ranges between barley and wheat confirmed that isotopes alone cannot perfectly separate the two species
- Neolithic samples clustered in the δ13C range of -26‰ to -22‰, while Iron Age samples extended to -20‰

**Geographical patterns:**
- Western Mediterranean samples showed more negative δ13C (wetter conditions)
- Eastern Mediterranean samples showed more positive δ13C (drier conditions) and higher δ15N (greater aridity)

**Correlations:**
- Weak negative correlation between δ13C and δ15N (-0.21), suggesting that water stress and aridity/ manuring are partially independent agricultural variables

### 3.4 Unsupervised Learning: K-means Clustering

**Objective:** Identify natural groupings of samples based only on isotopic values (δ13C, δ15N)

**Method:**
- K-means algorithm with k=4 (determined by elbow method and silhouette analysis)
- Features scaled with StandardScaler prior to clustering

**Cluster interpretation:**

| Cluster | δ13C (‰) | δ15N (‰) | n | Interpretation |
|:---|:---|:---|:---|:---|
| 0 | -24.5 | 4.2 | 408 | Good water availability, low aridity |
| 1 | -22.1 | 6.8 | 357 | Moderate water stress |
| 2 | -20.3 | 9.5 | 212 | High water stress, arid soils |
| 3 | -19.8 | 11.2 | 127 | Extreme aridity / manuring |

**Geographical distribution of clusters:**
- Cluster 0 (optimal conditions) dominated Western Mediterranean
- Cluster 3 (extreme stress) concentrated in Eastern Mediterranean
- This confirms that environmental gradients strongly influence isotopic signatures

### 3.5 Supervised Learning: Model Design and Comparison

#### Features selected for supervised models
| Feature | Type | Encoding |
|:---|:---|:---|
| IRMS_d13C_Collagen | Continuous | Scaled (StandardScaler) |
| d15N_Collagen | Continuous | Scaled (StandardScaler) |
| Latitude_N | Continuous | Scaled (StandardScaler) |
| Longitude_E | Continuous | Scaled (StandardScaler) |
| Chronological_Period_clean | Categorical | LabelEncoder |
| Mediterranean_Basin | Categorical | LabelEncoder |

**Target variable:** `Cereal_Type` (barley = 1, wheat = 0)

#### Best-results Models evaluated
1. **Random Forest (base)** - n_estimators=100, random_state=42
2. **XGBoost** - n_estimators=100, eval_metric='logloss'
3. **Random Forest (balanced)** - class_weight='balanced'

**Validation strategy:** 80/20 stratified train-test split

#### Results comparison

| Model | Accuracy | AUC | Top 1 Feature | Top 2 Feature | Top 3 Feature |
|:---|:---|:---|:---|:---|:---|
| Random Forest (base) | 74.35% | 0.825 | δ13C (42%) | δ15N (29%) | lat (12%) |
| XGBoost | 74.35% | 0.813 | lat (28%) | lon (22%) | δ13C (19%) |
| **Random Forest (balanced)** | **74.78%** | **0.824** | δ13C (41%) | δ15N (28%) | lat (13%) |

#### Model selection: Random Forest with class balancing

**Why Random Forest over XGBoost?**
- RF gives **71% combined importance to isotopes**, while XGBoost prioritises geography (44%)
- Isotopic relationships (δ13C × δ15N) are **non-linear and interactive** – RF's ensemble of parallel trees captures these interactions naturally
- XGBoost's sequential boosting optimises for **binary spatial divisions** (latitude/longitude thresholds), which is less relevant for this research question

**Why class balancing?**
- The dataset is slightly imbalanced (51.1% barley vs 48.9% wheat)
- `class_weight='balanced'` increases penalty for misclassifying the minority class (wheat)
- Result: accuracy improved by **+0.43%** with stable AUC

**Confusion matrix (RF balanced):**

| Actual \ Predicted | Wheat | Barley |
|:---|:---|:---|
| Wheat | 148 | 48 |
| Barley | 43 | 171 |

**Interpretation:** The model confuses wheat as barley slightly more often than the reverse, possibly due to isotopic overlap in certain environmental conditions.

#### Hyperparameter tuning (GridSearchCV)
Grid search was performed for Random Forest with the following parameters:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [10, 20, None]
- `min_samples_split`: [2, 5, 10]

**Best parameters:** n_estimators=100, max_depth=20, min_samples_split=2 (no significant improvement over base model)

### 3.6 Feature Importance Analysis

**Key insight:** The model does not rely on a single feature but combines multiple sources of information:

- **δ13C (41%):** Primary predictor – captures water stress, the main environmental variable differentiating cultivation conditions
- **δ15N (28%):** Secondary predictor – reflects soil aridity or manuring intensity
- **Latitude (13%):** Geographic proxy for climatic zone
- **Longitude (10%):** Proxy for Mediterranean sub-basin
- **Period (5%):** Chronological changes in agricultural practices
- **Basin (3%):** Regional agricultural traditions

This distribution confirms that **isotopic signals are the primary drivers of classification**, with geography and chronology providing complementary context.
